# Задание №5. Создание модели для рекомендации фильмов (коллаборативная фильтрация на основе эмбеддингов)

Данное задание представляет собой пошаговое руководство по созданию модели рекомендации фильмов на основе предпочтений пользователей. Вы познакомитесь с основными понятиями рекомендательных систем, научитесь подготавливать данные о рейтингах, строить и обучать модель матричной факторизации с использованием эмбеддингов пользователей и фильмов, а также использовать её для выдачи персонализированных рекомендаций с учётом жанра. Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и рекомендация) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия рекомендательных систем

### 1.1. Коллаборативная фильтрация (Collaborative Filtering)
Коллаборативная фильтрация – метод построения рекомендаций, основанный на предположении, что пользователи, которые схоже оценивали объекты в прошлом, будут иметь схожие предпочтения в будущем. В основе лежит матрица взаимодействий «пользователь-объект» (например, рейтингов фильмов).

### 1.2. Матричная факторизация (Matrix Factorization)
Один из самых популярных подходов к коллаборативной фильтрации. Исходная матрица рейтингов размера `(число пользователей) × (число фильмов)` аппроксимируется произведением двух низкоранговых матриц: матрицы скрытых факторов пользователей и матрицы скрытых факторов фильмов. Каждый пользователь и каждый фильм представляются вектором в **латентном пространстве** размерности `K`. Предсказанный рейтинг вычисляется как скалярное произведение соответствующих векторов, часто с добавлением смещений (bias) пользователя и фильма.

### 1.3. Эмбеддинги (Embeddings)
В контексте рекомендательных систем эмбеддинги – это обучаемые векторные представления категориальных признаков (идентификаторов пользователей и фильмов). Слой `Embedding` преобразует целочисленный идентификатор в плотный вектор фиксированной размерности. Эти векторы обучаются вместе с моделью так, чтобы скалярное произведение вектора пользователя и вектора фильма хорошо предсказывало реальный рейтинг.

### 1.4. Bias (смещение)
Добавление свободных членов (bias) позволяет учитывать систематические отклонения: например, некоторые пользователи ставят в среднем более высокие оценки, а некоторые фильмы всегда получают высокие оценки независимо от пользователя. Bias пользователя и фильма обучаются как отдельные эмбеддинги размерности 1.

### 1.5. Функция потерь и метрики
Для задачи предсказания рейтинга (регрессия) обычно используется **среднеквадратичная ошибка (MSE)** или **корень из среднеквадратичной ошибки (RMSE)**. Метрика RMSE интерпретируется в тех же единицах, что и рейтинг (например, если рейтинг от 1 до 5, RMSE показывает среднюю ошибку в баллах).

### 1.6. Оценка качества
Данные разделяются на обучающую и тестовую выборки. Модель обучается на обучающей выборке, а качество оценивается на тестовой по метрике RMSE. Чем меньше RMSE, тем точнее модель предсказывает рейтинги.

### 1.7. Генерация рекомендаций
Для конкретного пользователя модель может предсказать рейтинг для всех фильмов, которые он ещё не оценивал, и выбрать фильмы с наивысшими предсказаниями. Часто рекомендации дополнительно фильтруются по жанру, году и т.д.

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него файл с данными о рейтингах фильмов (`ratings.txt`), затем построить и обучить модель матричной факторизации для предсказания рейтингов. После обучения вы сохраните модель, а также кодировщики пользователей и фильмов, и загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для выдачи рекомендаций конкретному пользователю с фильтрацией по жанру без повторного обучения. Весь код и отчёт должны быть оформлены в одном Jupyter Notebook.

Формат файла `ratings.txt`: текстовый файл с разделителем `;`, каждая строка содержит поля: `User`, `Название`, `Жанр`, `Рейтинг`. Строки, начинающиеся с `#`, игнорируются (комментарии). Пример:
```
User;Название;Жанр;Рейтинг
Alice;Титаник;драма;5
Bob;Матрица;фантастика;4
# это комментарий
Charlie;Зеленая миля;драма;5
...
```

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи)
3. **Теоретическая часть** (объяснение ключевых понятий: коллаборативная фильтрация, матричная факторизация, эмбеддинги, bias, RMSE)
4. **Описание данных** (источник данных, статистика: количество пользователей, фильмов, оценок, распределение рейтингов; ссылка на файл в репозитории)
5. **Подготовка данных** (загрузка, кодирование идентификаторов, разделение на train/test)
6. **Построение модели** (архитектура: входы, эмбеддинги, скалярное произведение, bias, выход; визуализация модели, summary)
7. **Обучение модели** (параметры обучения, графики потерь на train/validation, расчёт RMSE на тесте)
8. **Функция рекомендаций** (описание функции `recommend`, примеры вызова для разных пользователей и жанров)
9. **Сохранение модели и вспомогательных объектов** (код сохранения модели, кодировщиков; загрузка файлов в репозиторий)
10. **Загрузка модели из репозитория и быстрые рекомендации** (код загрузки через raw‑ссылку, повторное использование для рекомендаций)
11. **Выводы** (что получилось, какие были трудности, возможные улучшения)
12. **Список использованных источников**
13. **Приложение** (полный код с комментариями)

---

## 4. Требования к данным и ссылка на репозиторий

### 4.1. Минимальный и максимальный объём данных
- **Минимальный объём**: не менее 1000 оценок, не менее 10 пользователей и 20 фильмов (чтобы модель могла выучить хоть какие-то закономерности).
- **Максимальный объём**: в Google Colab ограничение по оперативной памяти (обычно ~12–25 ГБ). Для комфортной работы рекомендуется использовать не более 1–2 миллионов оценок. При превышении можно уменьшить размерность эмбеддингов или использовать пакетную обработку.

### 4.2. Создание репозитория и загрузка данных
1. Зарегистрируйтесь на [GitHub](https://github.com) (если у вас ещё нет аккаунта).
2. Создайте новый публичный репозиторий с названием, например, `movie-recommender`.
3. Загрузите в репозиторий файл с данными `ratings.txt`. Вы можете использовать собственный датасет или сгенерировать синтетические данные с помощью кода ниже.  
   **Важно**: после загрузки получите **raw‑ссылку** на файл. Для этого откройте файл в репозитории, нажмите кнопку "Raw" и скопируйте URL. Он будет выглядеть примерно так:  
   `https://raw.githubusercontent.com/ваш_логин/movie-recommender/main/ratings.txt`

### 4.3. Синтетический датасет (если нет реального)
Если у вас нет собственных данных, вы можете сгенерировать синтетические оценки прямо в ноутбуке и затем сохранить их в файл для загрузки в репозиторий. Пример генерации:

```python
import random
import pandas as pd

users = [f"User{i}" for i in range(1, 51)]          # 50 пользователей
movies = [f"Movie{j}" for j in range(1, 101)]       # 100 фильмов
genres = ['драма', 'комедия', 'фантастика', 'боевик', 'мелодрама']

data = []
for user in users:
    # Каждый пользователь оценивает случайные 20–30 фильмов
    n_ratings = random.randint(20, 30)
    sampled_movies = random.sample(movies, n_ratings)
    for movie in sampled_movies:
        rating = random.randint(1, 5)
        genre = random.choice(genres)
        data.append([user, movie, genre, rating])

df = pd.DataFrame(data, columns=['User', 'Название', 'Жанр', 'Рейтинг'])
df.to_csv('ratings.txt', sep=';', index=False, encoding='utf-8')
```

Затем загрузите файл `ratings.txt` в свой репозиторий.

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

Импортируем необходимые библиотеки. В Google Colab они уже предустановлены.

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dot, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import requests
import io
```

### 5.2. Загрузка данных из репозитория

Используйте вашу raw‑ссылку на файл `ratings.txt`.

```python
# Вставьте вашу ссылку
url_data = "https://raw.githubusercontent.com/ваш_логин/movie-recommender/main/ratings.txt"

# Скачиваем файл
response = requests.get(url_data)
response.encoding = 'utf-8'
data = response.text

# Загружаем в DataFrame (разделитель ;, комментарии # игнорируем)
df = pd.read_csv(io.StringIO(data), sep=';', comment='#', encoding='utf-8')
print(f"Загружено оценок: {len(df)}")
print("Первые 5 строк:")
df.head()
```

### 5.3. Предобработка данных

Кодируем пользователей и фильмы с помощью `LabelEncoder`.

```python
user_enc = LabelEncoder()
movie_enc = LabelEncoder()

df['user_id'] = user_enc.fit_transform(df['User'])
df['movie_id'] = movie_enc.fit_transform(df['Название'])

n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()

# Словари для обратного преобразования (понадобятся позже)
user_names = dict(enumerate(user_enc.classes_))
movie_titles = dict(enumerate(movie_enc.classes_))
# Словарь жанров для каждого фильма (возьмём первое вхождение, предполагаем, что жанр фильма не меняется)
movie_genres = df.groupby('movie_id')['Жанр'].first().to_dict()

print(f"Число пользователей: {n_users}")
print(f"Число фильмов: {n_movies}")
print(f"Диапазон рейтингов: {df['Рейтинг'].min()} - {df['Рейтинг'].max()}")
```

### 5.4. Разделение на обучающую и тестовую выборки

Используем `train_test_split` для разделения данных (80% train, 20% test).

```python
train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Обучающих примеров: {len(train)}")
print(f"Тестовых примеров: {len(test)}")
```

### 5.5. Построение модели матричной факторизации

Определим архитектуру: входы для идентификаторов пользователя и фильма, эмбеддинги, скалярное произведение, bias-термы, выход – предсказанный рейтинг.

```python
K = 50  # размерность латентного пространства

# Входы
user_in = Input(shape=(1,), name='user_in')
movie_in = Input(shape=(1,), name='movie_in')

# Эмбеддинги пользователей и фильмов
user_emb = Embedding(input_dim=n_users, output_dim=K, name='user_emb')(user_in)
movie_emb = Embedding(input_dim=n_movies, output_dim=K, name='movie_emb')(movie_in)

# Превращаем в одномерные векторы
user_vec = Flatten()(user_emb)
movie_vec = Flatten()(movie_emb)

# Скалярное произведение
dot_product = Dot(axes=1)([user_vec, movie_vec])

# Смещения (bias)
user_bias = Embedding(input_dim=n_users, output_dim=1, name='user_bias')(user_in)
movie_bias = Embedding(input_dim=n_movies, output_dim=1, name='movie_bias')(movie_in)
user_b = Flatten()(user_bias)
movie_b = Flatten()(movie_bias)

# Итоговый прогноз: скалярное произведение + смещения
prediction = Add()([dot_product, user_b, movie_b])

model = Model(inputs=[user_in, movie_in], outputs=prediction)
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model.summary()
```

### 5.6. Обучение модели

Обучим модель с ранней остановкой для предотвращения переобучения.

```python
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    [train['user_id'], train['movie_id']],
    train['Рейтинг'].values,
    epochs=50,
    batch_size=64,
    validation_data=([test['user_id'], test['movie_id']], test['Рейтинг'].values),
    callbacks=[early_stop],
    verbose=1
)
```

### 5.7. Визуализация процесса обучения

Построим график потерь на обучающей и валидационной выборках.

```python
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери (MSE)')
plt.xlabel('Эпоха')
plt.ylabel('MSE')
plt.show()
```

### 5.8. Оценка качества на тестовой выборке

Вычислим RMSE – корень из среднеквадратичной ошибки.

```python
preds = model.predict([test['user_id'], test['movie_id']], verbose=0).flatten()
rmse = np.sqrt(mean_squared_error(test['Рейтинг'].values, preds))
print(f"RMSE на тестовой выборке: {rmse:.3f}")
```

### 5.9. Функция рекомендаций

Функция `recommend` принимает имя пользователя, жанр и количество фильмов. Она отбирает все фильмы указанного жанра, предсказывает для них рейтинг и возвращает топ-N.

```python
def recommend(user_name: str, genre: str, top_n: int = 5):
    if user_name not in user_enc.classes_:
        print(f"Пользователь «{user_name}» не найден.")
        return
    user_idx = user_enc.transform([user_name])[0]

    # Все фильмы нужного жанра
    candidates = [mid for mid, g in movie_genres.items() if g.lower() == genre.lower()]
    if not candidates:
        print(f"Фильмы жанра «{genre}» не найдены.")
        return

    # Предсказываем рейтинг для каждого кандидата
    user_arr = np.array([user_idx] * len(candidates))
    movie_arr = np.array(candidates)
    preds = model.predict([user_arr, movie_arr], verbose=0).flatten()

    # Отбираем топ-N
    top_indices = np.argsort(preds)[::-1][:top_n]
    recommended = [movie_titles[candidates[i]] for i in top_indices]

    print(f"\nРекомендованные фильмы в жанре «{genre.title()}» для пользователя «{user_name}»:")
    for title in recommended:
        print(f"- {title}")

# Пример вызова
recommend('User1', 'драма', 5)
```

### 5.10. Сохранение модели и вспомогательных объектов

Сохраним модель, кодировщики и словари. Эти файлы затем нужно загрузить в репозиторий.

```python
# Сохраняем модель Keras
model.save('movie_recommender.h5')
print("Модель сохранена в movie_recommender.h5")

# Сохраняем кодировщики и словари с помощью pickle
with open('user_encoder.pickle', 'wb') as f:
    pickle.dump(user_enc, f)
with open('movie_encoder.pickle', 'wb') as f:
    pickle.dump(movie_enc, f)
with open('movie_genres.pickle', 'wb') as f:
    pickle.dump(movie_genres, f)
with open('movie_titles.pickle', 'wb') as f:
    pickle.dump(movie_titles, f)

print("Вспомогательные объекты сохранены.")
```

### 5.11. Загрузка файлов в репозиторий

1. Скачайте файлы `movie_recommender.h5`, `user_encoder.pickle`, `movie_encoder.pickle`, `movie_genres.pickle`, `movie_titles.pickle` на свой компьютер (через панель файлов Colab).
2. В своём репозитории на GitHub нажмите "Add file" → "Upload files", загрузите эти файлы.
3. После загрузки получите raw‑ссылки на каждый файл. Для `.h5` лучше использовать ссылку вида `https://github.com/ваш_логин/movie-recommender/raw/main/movie_recommender.h5`, для pickle – аналогично.

### 5.12. Загрузка модели из репозитория и быстрые рекомендации

Этот блок можно выполнить после того, как вы загрузили файлы в репозиторий. Он демонстрирует, как загрузить сохранённые артефакты и использовать их для рекомендаций без повторного обучения.

```python
# Вставьте ваши ссылки
url_model = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_recommender.h5"
url_user_enc = "https://github.com/ваш_логин/movie-recommender/raw/main/user_encoder.pickle"
url_movie_enc = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_encoder.pickle"
url_genres = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_genres.pickle"
url_titles = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_titles.pickle"

# Скачиваем
!wget -O movie_recommender.h5 {url_model}
!wget -O user_encoder.pickle {url_user_enc}
!wget -O movie_encoder.pickle {url_movie_enc}
!wget -O movie_genres.pickle {url_genres}
!wget -O movie_titles.pickle {url_titles}

# Загружаем
loaded_model = load_model('movie_recommender.h5')
with open('user_encoder.pickle', 'rb') as f:
    loaded_user_enc = pickle.load(f)
with open('movie_encoder.pickle', 'rb') as f:
    loaded_movie_enc = pickle.load(f)
with open('movie_genres.pickle', 'rb') as f:
    loaded_movie_genres = pickle.load(f)
with open('movie_titles.pickle', 'rb') as f:
    loaded_movie_titles = pickle.load(f)

# Функция рекомендаций с загруженными объектами
def recommend_from_loaded(user_name, genre, top_n=5):
    if user_name not in loaded_user_enc.classes_:
        print(f"Пользователь «{user_name}» не найден.")
        return
    user_idx = loaded_user_enc.transform([user_name])[0]

    candidates = [mid for mid, g in loaded_movie_genres.items() if g.lower() == genre.lower()]
    if not candidates:
        print(f"Фильмы жанра «{genre}» не найдены.")
        return

    user_arr = np.array([user_idx] * len(candidates))
    movie_arr = np.array(candidates)
    preds = loaded_model.predict([user_arr, movie_arr], verbose=0).flatten()

    top_indices = np.argsort(preds)[::-1][:top_n]
    recommended = [loaded_movie_titles[candidates[i]] for i in top_indices]

    print(f"\nРекомендованные фильмы в жанре «{genre.title()}» для «{user_name}»:")
    for title in recommended:
        print(f"- {title}")

# Пример
recommend_from_loaded('User1', 'драма', 5)
```

### 5.13. Интерактивный режим (по желанию)

Можно добавить цикл для ввода имени пользователя и жанра.

```python
print("=== Рекомендательная система (для выхода введите пустую строку) ===")
while True:
    user = input("Введите имя пользователя: ").strip()
    if not user:
        break
    genre = input("Введите жанр: ").strip()
    if not genre:
        break
    recommend_from_loaded(user, genre, 5)
```

---

## 6. Полный код нейронной сети с обучением

Для удобства ниже приведён сводный код всех ячеек, необходимых для полного цикла обучения модели (без интерактива). Вы можете скопировать его в одну ячейку или выполнять по частям.

```python
# Импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dot, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import requests
import io

# Загрузка данных из репозитория
url_data = "https://raw.githubusercontent.com/ваш_логин/movie-recommender/main/ratings.txt"
response = requests.get(url_data)
response.encoding = 'utf-8'
df = pd.read_csv(io.StringIO(response.text), sep=';', comment='#', encoding='utf-8')
print(f"Загружено оценок: {len(df)}")

# Кодирование
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
df['user_id'] = user_enc.fit_transform(df['User'])
df['movie_id'] = movie_enc.fit_transform(df['Название'])

n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()
movie_titles = dict(enumerate(movie_enc.classes_))
movie_genres = df.groupby('movie_id')['Жанр'].first().to_dict()

print(f"Пользователей: {n_users}, фильмов: {n_movies}")

# Разделение
train, test = train_test_split(df, test_size=0.2, random_state=42)

# Модель
K = 50
user_in = Input(shape=(1,))
movie_in = Input(shape=(1,))
u_emb = Embedding(n_users, K)(user_in)
m_emb = Embedding(n_movies, K)(movie_in)
u_vec = Flatten()(u_emb)
m_vec = Flatten()(m_emb)
dot = Dot(axes=1)([u_vec, m_vec])
u_bias = Embedding(n_users, 1)(user_in)
m_bias = Embedding(n_movies, 1)(movie_in)
u_b = Flatten()(u_bias)
m_b = Flatten()(m_bias)
pred = Add()([dot, u_b, m_b])

model = Model([user_in, movie_in], pred)
model.compile(optimizer=Adam(0.001), loss='mse')
model.summary()

# Обучение
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    [train['user_id'], train['movie_id']], train['Рейтинг'],
    epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stop], verbose=1
)

# График
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('MSE')
plt.show()

# Оценка
preds = model.predict([test['user_id'], test['movie_id']], verbose=0).flatten()
rmse = np.sqrt(mean_squared_error(test['Рейтинг'], preds))
print(f"RMSE на тесте: {rmse:.3f}")

# Сохранение
model.save('movie_recommender.h5')
with open('user_encoder.pickle', 'wb') as f: pickle.dump(user_enc, f)
with open('movie_encoder.pickle', 'wb') as f: pickle.dump(movie_enc, f)
with open('movie_genres.pickle', 'wb') as f: pickle.dump(movie_genres, f)
with open('movie_titles.pickle', 'wb') as f: pickle.dump(movie_titles, f)
print("Модель и объекты сохранены.")
```

---

## 7. Код с загруженной моделью для быстрого запуска

Этот блок можно использовать отдельно, если модель уже сохранена в репозитории. Он загружает файлы по raw‑ссылкам и выполняет рекомендации.

```python
# Укажите ваши ссылки
url_model = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_recommender.h5"
url_user_enc = "https://github.com/ваш_логин/movie-recommender/raw/main/user_encoder.pickle"
url_movie_enc = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_encoder.pickle"
url_genres = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_genres.pickle"
url_titles = "https://github.com/ваш_логин/movie-recommender/raw/main/movie_titles.pickle"

!wget -O movie_recommender.h5 {url_model}
!wget -O user_encoder.pickle {url_user_enc}
!wget -O movie_encoder.pickle {url_movie_enc}
!wget -O movie_genres.pickle {url_genres}
!wget -O movie_titles.pickle {url_titles}

import numpy as np
import pickle
from tensorflow.keras.models import load_model

loaded_model = load_model('movie_recommender.h5')
with open('user_encoder.pickle', 'rb') as f: user_enc = pickle.load(f)
with open('movie_encoder.pickle', 'rb') as f: movie_enc = pickle.load(f)
with open('movie_genres.pickle', 'rb') as f: movie_genres = pickle.load(f)
with open('movie_titles.pickle', 'rb') as f: movie_titles = pickle.load(f)

def recommend(user_name, genre, top_n=5):
    if user_name not in user_enc.classes_:
        print("Пользователь не найден")
        return
    u_idx = user_enc.transform([user_name])[0]
    candidates = [mid for mid, g in movie_genres.items() if g.lower() == genre.lower()]
    if not candidates:
        print("Жанр не найден")
        return
    u_arr = np.array([u_idx]*len(candidates))
    m_arr = np.array(candidates)
    preds = loaded_model.predict([u_arr, m_arr], verbose=0).flatten()
    top = np.argsort(preds)[::-1][:top_n]
    for i in top:
        print(f"- {movie_titles[candidates[i]]}")

# Пример
recommend('User1', 'драма', 5)
```

---

## 8. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Подготовьте файл `ratings.txt` с оценками фильмов (реальный или синтетический). Убедитесь, что объём данных соответствует требованиям (минимум 1000 оценок, не менее 10 пользователей и 20 фильмов). Загрузите его в репозиторий.
- Получите raw‑ссылку на файл и запишите её для использования в ноутбуке.

### Задание 2. Подготовка данных в ноутбуке
- В ноутбуке напишите код для загрузки данных из репозитория по ссылке.
- Выведите статистику: количество оценок, число пользователей, число фильмов, распределение рейтингов (гистограмма).
- Закодируйте пользователей и фильмы с помощью `LabelEncoder`. Объясните, зачем это нужно.
- Разделите данные на обучающую и тестовую выборки (80/20).

### Задание 3. Построение модели
- Постройте модель матричной факторизации с размерностью латентного пространства `K = 50` (можно экспериментировать с другими значениями).
- Добавьте bias-термы для пользователей и фильмов.
- Выведите summary модели.
- Скомпилируйте модель с оптимизатором Adam и функцией потерь MSE.

### Задание 4. Обучение модели
- Обучите модель с использованием `EarlyStopping` (patience=5) на 50 эпохах.
- Постройте график потерь на обучающей и валидационной выборках.
- Вычислите RMSE на тестовой выборке и интерпретируйте результат.

### Задание 5. Функция рекомендаций (с текущей моделью)
- Реализуйте функцию `recommend(user_name, genre, top_n)`, которая выводит топ-N фильмов указанного жанра с наивысшими предсказанными рейтингами для данного пользователя.
- Протестируйте её для 2–3 разных пользователей и 2–3 жанров. Приведите примеры вывода.

### Задание 6. Сохранение и загрузка в репозиторий
- Сохраните обученную модель в файл `movie_recommender.h5`.
- Сохраните кодировщики пользователей и фильмов, а также словари `movie_titles` и `movie_genres` с помощью `pickle`.
- Загрузите все эти файлы в свой репозиторий на GitHub.
- Получите raw‑ссылки на каждый файл.

### Задание 7. Загрузка модели из репозитория и быстрые рекомендации
- В отдельной ячейке напишите код, который скачивает файлы модели и вспомогательных объектов из репозитория по ссылкам.
- Загрузите модель и объекты, и с их помощью вызовите функцию рекомендаций для тех же пользователей и жанров, что и в задании 5. Убедитесь, что результаты совпадают (с учётом возможных погрешностей).

### Задание 8. Анализ результатов
- Оцените, насколько хорошо модель предсказывает рейтинги (RMSE). Что можно считать приемлемым значением для вашего датасета?
- Попробуйте изменить размерность эмбеддингов (`K`) и посмотрите, как меняется RMSE.
- Предложите способы улучшения модели (например, добавление дополнительных признаков, использование глубинных архитектур, учёт времени и т.д.).

### Задание 9*. Дополнительно (по желанию)
- Реализуйте рекомендации без фильтра по жанру (просто топ-N фильмов с наибольшим предсказанным рейтингом).
- Добавьте возможность исключать фильмы, которые пользователь уже оценил.
- Попробуйте использовать другую функцию потерь (например, MAE) и сравните результаты.
- Визуализируйте эмбеддинги фильмов с помощью t-SNE или PCA и попробуйте интерпретировать кластеры.

---

## 9. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с основными понятиями рекомендательных систем и коллаборативной фильтрации;
- подготовили данные о рейтингах, закодировали категориальные признаки;
- построили и обучили модель матричной факторизации с эмбеддингами и bias-термами;
- оценили качество модели по метрике RMSE;
- реализовали функцию персонализированных рекомендаций с учётом жанра;
- освоили сохранение модели и вспомогательных объектов, а затем их загрузку из репозитория для повторного использования.

Полученные навыки являются основой для построения более сложных рекомендательных систем, включая нейросетевые архитектуры с дополнительными признаками (возраст, пол, теги фильмов) и гибридные подходы.